# Flash Attention: Fast and Memory-Efficient Exact Attention

**Historical Context**: Flash Attention was introduced in 2022 by Tri Dao et al. at Stanford to address the quadratic memory bottleneck in standard attention mechanisms. Flash Attention v2 followed in 2023 with further optimizations.

**Key Innovation**: By carefully reordering attention computation and leveraging GPU memory hierarchy (SRAM vs HBM), Flash Attention achieves exact attention with O(N) memory instead of O(N²) while being 2-4x faster.

**Learning Objectives**:
- Understand the O(N²) memory problem in standard attention
- Learn the Flash Attention algorithm with tiling and recomputation
- Compare standard vs Flash Attention implementations
- Benchmark memory usage and speed improvements
- Know when to use Flash Attention v1 vs v2
- Integrate Flash Attention with transformers

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
from typing import Optional, Tuple
import math

# Check if flash attention is available
try:
    from flash_attn import flash_attn_func
    FLASH_AVAILABLE = True
except ImportError:
    FLASH_AVAILABLE = False
    print("Flash Attention not installed. Install with: pip install flash-attn --no-build-isolation")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 1. The O(N²) Memory Problem in Standard Attention

Standard attention computes:
```
S = QK^T / sqrt(d)     # O(N²) memory for scores
P = softmax(S)         # O(N²) memory for probabilities
O = PV                 # Final output
```

For sequence length N=4096 and batch size B=8 with 32 heads:
- Attention matrix: 8 × 32 × 4096 × 4096 × 4 bytes = 16 GB
- This is stored in slow HBM (High Bandwidth Memory)
- Backward pass requires storing S and P, doubling memory

In [ ]:
def standard_attention(Q, K, V, mask=None):
    """
    Standard attention implementation.
    
    Args:
        Q: (batch, heads, seq_len, head_dim)
        K: (batch, heads, seq_len, head_dim)
        V: (batch, heads, seq_len, head_dim)
        mask: Optional attention mask
    
    Returns:
        output: (batch, heads, seq_len, head_dim)
        attention_weights: (batch, heads, seq_len, seq_len)
    """
    batch, heads, seq_len, head_dim = Q.shape
    
    # Compute attention scores: O(N²) memory
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(head_dim)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax: stores O(N²) probabilities
    attn_weights = F.softmax(scores, dim=-1)
    
    # Weighted sum
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights

# Demonstrate memory usage
def measure_memory(batch_size, num_heads, seq_len, head_dim):
    """Calculate theoretical memory for attention matrix."""
    # Attention matrix size in bytes (float32 = 4 bytes)
    attn_memory_mb = (batch_size * num_heads * seq_len * seq_len * 4) / (1024**2)
    
    # For backward pass, need to store both scores and softmax output
    backward_memory_mb = attn_memory_mb * 2
    
    return attn_memory_mb, backward_memory_mb

# Example calculations
configs = [
    (1, 8, 512, 64),
    (1, 8, 1024, 64),
    (1, 8, 2048, 64),
    (1, 8, 4096, 64),
    (8, 32, 4096, 64),
]

print("Memory Usage for Attention Matrix:")
print("="*70)
print(f"{'Config':<25} {'Forward (MB)':<15} {'Backward (MB)':<15} {'Total (GB)'}")
print("="*70)

for batch, heads, seq_len, head_dim in configs:
    fwd_mem, bwd_mem = measure_memory(batch, heads, seq_len, head_dim)
    total_gb = (fwd_mem + bwd_mem) / 1024
    config_str = f"B={batch}, H={heads}, N={seq_len}"
    print(f"{config_str:<25} {fwd_mem:<15.2f} {bwd_mem:<15.2f} {total_gb:<15.2f}")

print("\nNote: This only accounts for the attention matrix, not Q, K, V tensors!")

## 2. Flash Attention Algorithm: Tiling and Recomputation

**Key Ideas**:
1. **Tiling**: Break Q, K, V into blocks that fit in fast SRAM
2. **Fused Operations**: Compute softmax in a single kernel pass
3. **Recomputation**: In backward pass, recompute attention on-the-fly instead of storing
4. **Online Softmax**: Use numerically stable softmax with running statistics

**Algorithm**:
```
For each block of Q:
    For each block of K, V:
        - Load blocks into SRAM
        - Compute local attention
        - Update output with online softmax
        - Discard intermediate values
```

**Memory Complexity**: O(N) instead of O(N²)

In [ ]:
def naive_flash_attention_simulation(Q, K, V, block_size=64):
    """
    Simplified simulation of Flash Attention algorithm (not optimized).
    This demonstrates the concept but doesn't achieve actual speedup.
    
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        block_size: Size of blocks for tiling
    """
    batch, heads, seq_len, head_dim = Q.shape
    scale = 1.0 / math.sqrt(head_dim)
    
    # Initialize output and normalization statistics
    O = torch.zeros_like(Q)
    l = torch.zeros(batch, heads, seq_len, 1, device=Q.device)
    m = torch.full((batch, heads, seq_len, 1), float('-inf'), device=Q.device)
    
    # Iterate over blocks of Q (rows)
    for i in range(0, seq_len, block_size):
        i_end = min(i + block_size, seq_len)
        Q_block = Q[:, :, i:i_end, :]
        
        # Initialize block output
        O_block = torch.zeros_like(Q_block)
        l_block = torch.zeros(batch, heads, i_end - i, 1, device=Q.device)
        m_block = torch.full((batch, heads, i_end - i, 1), float('-inf'), device=Q.device)
        
        # Iterate over blocks of K, V (columns)
        for j in range(0, seq_len, block_size):
            j_end = min(j + block_size, seq_len)
            K_block = K[:, :, j:j_end, :]
            V_block = V[:, :, j:j_end, :]
            
            # Compute attention scores for this block
            S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) * scale
            
            # Online softmax: update running max
            m_new = torch.maximum(m_block, S_block.max(dim=-1, keepdim=True)[0])
            
            # Compute exponentials with numerical stability
            exp_S = torch.exp(S_block - m_new)
            exp_m_old = torch.exp(m_block - m_new)
            
            # Update normalization term
            l_new = exp_m_old * l_block + exp_S.sum(dim=-1, keepdim=True)
            
            # Update output
            O_block = (exp_m_old * l_block * O_block + torch.matmul(exp_S, V_block)) / l_new
            
            # Update statistics
            m_block = m_new
            l_block = l_new
        
        # Store block output
        O[:, :, i:i_end, :] = O_block
    
    return O

# Test correctness
batch, heads, seq_len, head_dim = 2, 4, 128, 32
Q = torch.randn(batch, heads, seq_len, head_dim, device=device)
K = torch.randn(batch, heads, seq_len, head_dim, device=device)
V = torch.randn(batch, heads, seq_len, head_dim, device=device)

# Standard attention
O_standard, _ = standard_attention(Q, K, V)

# Naive flash attention
O_flash_sim = naive_flash_attention_simulation(Q, K, V, block_size=32)

# Check correctness
max_diff = torch.abs(O_standard - O_flash_sim).max().item()
mean_diff = torch.abs(O_standard - O_flash_sim).mean().item()

print(f"Correctness Check:")
print(f"Max difference: {max_diff:.6f}")
print(f"Mean difference: {mean_diff:.6f}")
print(f"Are outputs close? {torch.allclose(O_standard, O_flash_sim, atol=1e-5)}")

## 3. Visualizing Memory Hierarchy and Tiling

GPU Memory Hierarchy:
- **HBM (High Bandwidth Memory)**: Large (40GB) but slow (1.5TB/s)
- **SRAM (On-chip memory)**: Small (20MB) but fast (19TB/s)

Flash Attention exploits this by keeping frequently accessed data in SRAM.

In [ ]:
def visualize_attention_tiling():
    """Visualize how Flash Attention tiles the attention computation."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Standard attention
    ax = axes[0]
    seq_len = 8
    attn_matrix = np.random.rand(seq_len, seq_len)
    im = ax.imshow(attn_matrix, cmap='Blues', aspect='auto')
    ax.set_title('Standard Attention\n(Full N×N matrix in memory)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Key/Value position')
    ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax, label='Attention weight')
    
    # Flash attention with tiling
    ax = axes[1]
    block_size = 2
    attn_matrix = np.random.rand(seq_len, seq_len)
    im = ax.imshow(attn_matrix, cmap='Blues', aspect='auto', alpha=0.3)
    
    # Draw block boundaries
    for i in range(0, seq_len, block_size):
        for j in range(0, seq_len, block_size):
            rect = plt.Rectangle((j-0.5, i-0.5), block_size, block_size,
                                fill=False, edgecolor='red', linewidth=2)
            ax.add_patch(rect)
    
    # Highlight current block being processed
    current_i, current_j = 2, 4
    rect = plt.Rectangle((current_j-0.5, current_i-0.5), block_size, block_size,
                         fill=True, facecolor='orange', alpha=0.5, edgecolor='red', linewidth=3)
    ax.add_patch(rect)
    
    ax.set_title('Flash Attention\n(Only process one block at a time)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Key/Value position')
    ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax, label='Attention weight')
    ax.text(current_j + block_size/2, current_i + block_size/2, 'Current\nBlock',
            ha='center', va='center', color='white', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/flash_attention_tiling.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_attention_tiling()

# Visualize memory bandwidth comparison
def visualize_memory_hierarchy():
    fig, ax = plt.subplots(figsize=(10, 6))
    
    memory_types = ['SRAM\n(On-chip)', 'HBM\n(GPU Memory)', 'CPU RAM', 'SSD Storage']
    sizes_gb = [0.02, 40, 128, 1000]  # Typical sizes in GB
    bandwidth_tbs = [19, 1.5, 0.1, 0.005]  # Bandwidth in TB/s
    
    x = np.arange(len(memory_types))
    width = 0.35
    
    ax1 = ax
    color1 = 'tab:blue'
    ax1.set_ylabel('Bandwidth (TB/s)', color=color1, fontsize=12)
    bars1 = ax1.bar(x - width/2, bandwidth_tbs, width, label='Bandwidth', color=color1, alpha=0.7)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.set_yscale('log')
    
    ax2 = ax1.twinx()
    color2 = 'tab:orange'
    ax2.set_ylabel('Capacity (GB)', color=color2, fontsize=12)
    bars2 = ax2.bar(x + width/2, sizes_gb, width, label='Capacity', color=color2, alpha=0.7)
    ax2.tick_params(axis='y', labelcolor=color2)
    ax2.set_yscale('log')
    
    ax1.set_xlabel('Memory Type', fontsize=12)
    ax1.set_title('GPU Memory Hierarchy: Size vs Speed Trade-off', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(memory_types)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add annotations
    ax1.annotate('Flash Attention\noptimizes here!', xy=(0, bandwidth_tbs[0]),
                xytext=(0.5, 10), fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    
    fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/memory_hierarchy.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_memory_hierarchy()

## 4. Standard vs Flash Attention Benchmarks

We'll compare:
1. Memory usage (forward and backward)
2. Speed (forward and backward)
3. Scalability with sequence length

In [ ]:
def benchmark_attention(batch_size, num_heads, seq_lengths, head_dim=64, num_runs=10):
    """
    Benchmark standard vs Flash Attention across different sequence lengths.
    """
    results = {
        'seq_len': [],
        'standard_fwd_time': [],
        'standard_bwd_time': [],
        'standard_memory': [],
        'flash_fwd_time': [],
        'flash_bwd_time': [],
        'flash_memory': [],
    }
    
    for seq_len in seq_lengths:
        print(f"\nBenchmarking sequence length: {seq_len}")
        results['seq_len'].append(seq_len)
        
        # Standard attention
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        
        Q = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, requires_grad=True)
        K = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, requires_grad=True)
        V = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, requires_grad=True)
        
        # Warmup
        for _ in range(3):
            O, _ = standard_attention(Q, K, V)
            O.sum().backward()
        
        # Forward timing
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(num_runs):
            O, _ = standard_attention(Q, K, V)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        fwd_time = (time.time() - start) / num_runs * 1000
        results['standard_fwd_time'].append(fwd_time)
        
        # Backward timing
        O, _ = standard_attention(Q, K, V)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(num_runs):
            if Q.grad is not None:
                Q.grad.zero_()
                K.grad.zero_()
                V.grad.zero_()
            O.sum().backward(retain_graph=True)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        bwd_time = (time.time() - start) / num_runs * 1000
        results['standard_bwd_time'].append(bwd_time)
        
        # Memory
        if device.type == 'cuda':
            memory_mb = torch.cuda.max_memory_allocated() / (1024**2)
            results['standard_memory'].append(memory_mb)
        else:
            results['standard_memory'].append(0)
        
        print(f"  Standard - Forward: {fwd_time:.2f}ms, Backward: {bwd_time:.2f}ms, Memory: {results['standard_memory'][-1]:.2f}MB")
        
        # Flash Attention (if available)
        if FLASH_AVAILABLE and device.type == 'cuda':
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            
            # Flash attention expects (batch, seq_len, num_heads, head_dim)
            Q_flash = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device, requires_grad=True)
            K_flash = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device, requires_grad=True)
            V_flash = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device, requires_grad=True)
            
            # Warmup
            for _ in range(3):
                O_flash = flash_attn_func(Q_flash, K_flash, V_flash)
                O_flash.sum().backward()
            
            # Forward timing
            torch.cuda.synchronize()
            start = time.time()
            for _ in range(num_runs):
                O_flash = flash_attn_func(Q_flash, K_flash, V_flash)
            torch.cuda.synchronize()
            fwd_time = (time.time() - start) / num_runs * 1000
            results['flash_fwd_time'].append(fwd_time)
            
            # Backward timing
            O_flash = flash_attn_func(Q_flash, K_flash, V_flash)
            torch.cuda.synchronize()
            start = time.time()
            for _ in range(num_runs):
                if Q_flash.grad is not None:
                    Q_flash.grad.zero_()
                    K_flash.grad.zero_()
                    V_flash.grad.zero_()
                O_flash.sum().backward(retain_graph=True)
            torch.cuda.synchronize()
            bwd_time = (time.time() - start) / num_runs * 1000
            results['flash_bwd_time'].append(bwd_time)
            
            memory_mb = torch.cuda.max_memory_allocated() / (1024**2)
            results['flash_memory'].append(memory_mb)
            
            print(f"  Flash    - Forward: {fwd_time:.2f}ms, Backward: {bwd_time:.2f}ms, Memory: {memory_mb:.2f}MB")
            speedup_fwd = results['standard_fwd_time'][-1] / fwd_time
            speedup_bwd = results['standard_bwd_time'][-1] / bwd_time
            memory_reduction = results['standard_memory'][-1] / memory_mb
            print(f"  Speedup - Forward: {speedup_fwd:.2f}x, Backward: {speedup_bwd:.2f}x, Memory reduction: {memory_reduction:.2f}x")
        else:
            results['flash_fwd_time'].append(None)
            results['flash_bwd_time'].append(None)
            results['flash_memory'].append(None)
    
    return results

# Run benchmarks
if device.type == 'cuda':
    seq_lengths = [512, 1024, 2048, 4096]
else:
    seq_lengths = [128, 256, 512]  # Smaller for CPU

print("Running benchmarks...")
print("="*70)
benchmark_results = benchmark_attention(
    batch_size=2,
    num_heads=8,
    seq_lengths=seq_lengths,
    head_dim=64,
    num_runs=10 if device.type == 'cuda' else 3
)

In [ ]:
# Visualize benchmark results
def plot_benchmark_results(results):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    seq_lens = results['seq_len']
    
    # Forward time
    ax = axes[0, 0]
    ax.plot(seq_lens, results['standard_fwd_time'], 'o-', label='Standard', linewidth=2, markersize=8)
    if results['flash_fwd_time'][0] is not None:
        ax.plot(seq_lens, results['flash_fwd_time'], 's-', label='Flash Attention', linewidth=2, markersize=8)
    ax.set_xlabel('Sequence Length', fontsize=11)
    ax.set_ylabel('Forward Time (ms)', fontsize=11)
    ax.set_title('Forward Pass Speed', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    
    # Backward time
    ax = axes[0, 1]
    ax.plot(seq_lens, results['standard_bwd_time'], 'o-', label='Standard', linewidth=2, markersize=8)
    if results['flash_bwd_time'][0] is not None:
        ax.plot(seq_lens, results['flash_bwd_time'], 's-', label='Flash Attention', linewidth=2, markersize=8)
    ax.set_xlabel('Sequence Length', fontsize=11)
    ax.set_ylabel('Backward Time (ms)', fontsize=11)
    ax.set_title('Backward Pass Speed', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    
    # Memory usage
    ax = axes[1, 0]
    ax.plot(seq_lens, results['standard_memory'], 'o-', label='Standard', linewidth=2, markersize=8)
    if results['flash_memory'][0] is not None:
        ax.plot(seq_lens, results['flash_memory'], 's-', label='Flash Attention', linewidth=2, markersize=8)
    ax.set_xlabel('Sequence Length', fontsize=11)
    ax.set_ylabel('Peak Memory (MB)', fontsize=11)
    ax.set_title('Memory Usage', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    
    # Speedup comparison
    ax = axes[1, 1]
    if results['flash_fwd_time'][0] is not None:
        speedup_fwd = [s / f for s, f in zip(results['standard_fwd_time'], results['flash_fwd_time'])]
        speedup_bwd = [s / f for s, f in zip(results['standard_bwd_time'], results['flash_bwd_time'])]
        memory_reduction = [s / f for s, f in zip(results['standard_memory'], results['flash_memory'])]
        
        x = np.arange(len(seq_lens))
        width = 0.25
        
        ax.bar(x - width, speedup_fwd, width, label='Forward Speedup', alpha=0.8)
        ax.bar(x, speedup_bwd, width, label='Backward Speedup', alpha=0.8)
        ax.bar(x + width, memory_reduction, width, label='Memory Reduction', alpha=0.8)
        
        ax.axhline(y=1, color='red', linestyle='--', linewidth=2, label='Baseline')
        ax.set_xlabel('Sequence Length', fontsize=11)
        ax.set_ylabel('Speedup / Reduction Factor', fontsize=11)
        ax.set_title('Flash Attention Improvements', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([str(s) for s in seq_lens])
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Flash Attention\nNot Available', 
               ha='center', va='center', transform=ax.transAxes, fontsize=14)
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/flash_attention_benchmarks.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_benchmark_results(benchmark_results)

## 5. Flash Attention v1 vs v2

**Flash Attention v1** (2022):
- Original algorithm with tiling and recomputation
- ~2-4x speedup, O(N) memory
- Works well for A100 GPUs

**Flash Attention v2** (2023):
- Better work partitioning across thread blocks
- Reduced non-matmul operations
- Better utilization of GPU SMs (Streaming Multiprocessors)
- ~2x faster than v1 (4-8x vs standard)
- Optimized for H100 GPUs

**When to use which**:
- Use v2 when available (default in latest flash-attn package)
- v1 might be needed for older GPU architectures
- Both provide same numerical results

In [ ]:
# Comparison table
import pandas as pd

comparison_data = {
    'Feature': [
        'Memory Complexity',
        'Forward Speed',
        'Backward Speed',
        'GPU Utilization',
        'Sequence Length',
        'Numerical Accuracy',
        'Implementation',
        'Best GPU',
    ],
    'Standard Attention': [
        'O(N²)',
        'Baseline',
        'Baseline',
        'Low (many HBM reads)',
        'Limited by memory',
        'Exact',
        'PyTorch native',
        'Any',
    ],
    'Flash Attention v1': [
        'O(N)',
        '2-4x faster',
        '2-4x faster',
        'Medium',
        'Up to 64K',
        'Exact',
        'CUDA kernel',
        'A100',
    ],
    'Flash Attention v2': [
        'O(N)',
        '4-8x faster',
        '4-8x faster',
        'High (better parallelism)',
        'Up to 128K',
        'Exact',
        'Optimized CUDA',
        'H100',
    ],
}

df = pd.DataFrame(comparison_data)
print("\nFlash Attention Comparison:")
print("="*100)
print(df.to_string(index=False))
print("="*100)

## 6. Integration with Transformers

Flash Attention can be easily integrated into transformer models:

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head attention with optional Flash Attention support.
    """
    def __init__(self, d_model, num_heads, dropout=0.1, use_flash=False):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.use_flash = use_flash and FLASH_AVAILABLE
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: Optional attention mask
        
        Returns:
            output: (batch, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        Q = self.W_q(x)  # (batch, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        if self.use_flash:
            # Flash Attention expects (batch, seq_len, num_heads, head_dim)
            Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim)
            K = K.view(batch_size, seq_len, self.num_heads, self.head_dim)
            V = V.view(batch_size, seq_len, self.num_heads, self.head_dim)
            
            # Apply Flash Attention
            attn_output = flash_attn_func(Q, K, V, dropout_p=self.dropout.p if self.training else 0.0)
            # Output: (batch, seq_len, num_heads, head_dim)
            
            # Reshape back
            attn_output = attn_output.contiguous().view(batch_size, seq_len, self.d_model)
        else:
            # Standard attention: reshape to (batch, num_heads, seq_len, head_dim)
            Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
            K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
            V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
            
            # Scaled dot-product attention
            scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
            
            if mask is not None:
                scores = scores.masked_fill(mask == 0, float('-inf'))
            
            attn_weights = F.softmax(scores, dim=-1)
            attn_weights = self.dropout(attn_weights)
            
            attn_output = torch.matmul(attn_weights, V)
            # (batch, num_heads, seq_len, head_dim)
            
            # Reshape back
            attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # Final projection
        output = self.W_o(attn_output)
        
        return output

# Example usage
d_model = 512
num_heads = 8
batch_size = 4
seq_len = 128

x = torch.randn(batch_size, seq_len, d_model, device=device)

# Standard attention
mha_standard = MultiHeadAttention(d_model, num_heads, use_flash=False).to(device)
output_standard = mha_standard(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output_standard.shape}")
print(f"\nUsing Flash Attention: {FLASH_AVAILABLE}")

if FLASH_AVAILABLE:
    mha_flash = MultiHeadAttention(d_model, num_heads, use_flash=True).to(device)
    # Copy weights for fair comparison
    mha_flash.load_state_dict(mha_standard.state_dict())
    output_flash = mha_flash(x)
    print(f"Flash output shape: {output_flash.shape}")

## 7. Using Flash Attention with Hugging Face Transformers

Many Hugging Face models now support Flash Attention out of the box:

In [ ]:
# Example: Using Flash Attention with Hugging Face
example_code = '''
from transformers import AutoModelForCausalLM, AutoTokenizer

# Method 1: Enable Flash Attention v2 at model load time
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",  # Enable Flash Attention v2
    device_map="auto"
)

# Method 2: Enable SDPA (Scaled Dot Product Attention) - PyTorch 2.0+
# This uses PyTorch's native fused attention (includes Flash Attention)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",  # Use PyTorch SDPA
    device_map="auto"
)

# Method 3: For custom models
from transformers.models.llama.modeling_llama import LlamaConfig

config = LlamaConfig.from_pretrained("meta-llama/Llama-2-7b-hf")
config._attn_implementation = "flash_attention_2"
model = AutoModelForCausalLM.from_config(config)

# Supported models (as of 2024):
# - Llama, Llama 2, Llama 3
# - Mistral, Mixtral
# - Falcon
# - GPT-NeoX
# - Phi
# - And many more...
'''

print("Flash Attention Integration with Hugging Face:")
print("="*70)
print(example_code)

## 8. Practical Guidelines and Best Practices

### When to Use Flash Attention:

1. **Long Sequences** (N > 512):
   - Memory savings become significant
   - Speed improvements are most noticeable

2. **Memory-Constrained Training**:
   - Enables larger batch sizes
   - Allows longer context windows

3. **Large Models**:
   - Multi-head attention with many heads
   - Deep transformer models (many layers)

### When Standard Attention Might Be Better:

1. **Short Sequences** (N < 256):
   - Overhead of Flash Attention may not be worth it
   - Standard attention is simpler

2. **Custom Attention Patterns**:
   - Sparse attention patterns
   - Attention biases or custom masks
   - Flash Attention requires full attention

3. **Debugging**:
   - Standard attention is easier to inspect
   - Attention weights are directly accessible

### Installation and Compatibility:

```bash
# Flash Attention v2 (recommended)
pip install flash-attn --no-build-isolation

# Requirements:
# - CUDA 11.4+
# - PyTorch 1.12+
# - GPU: A100, A6000, RTX 3090/4090, H100
# - Compute capability >= 8.0 (Ampere or newer)
```

In [ ]:
# Summary visualization
def create_summary_comparison():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    categories = ['Memory\nComplexity', 'Forward\nSpeed', 'Backward\nSpeed', 
                  'Max Seq\nLength', 'GPU\nUtilization']
    
    # Normalize to standard attention = 1.0
    standard = [1.0, 1.0, 1.0, 1.0, 1.0]
    flash_v1 = [4.0, 2.5, 2.5, 4.0, 2.0]  # Improvements over standard
    flash_v2 = [4.0, 5.0, 5.0, 8.0, 3.5]
    
    x = np.arange(len(categories))
    width = 0.25
    
    bars1 = ax.bar(x - width, standard, width, label='Standard', alpha=0.8, color='#FF6B6B')
    bars2 = ax.bar(x, flash_v1, width, label='Flash Attention v1', alpha=0.8, color='#4ECDC4')
    bars3 = ax.bar(x + width, flash_v2, width, label='Flash Attention v2', alpha=0.8, color='#45B7D1')
    
    ax.set_ylabel('Improvement Factor (vs Standard)', fontsize=12)
    ax.set_title('Flash Attention: Overall Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=10)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.axhline(y=1, color='black', linestyle='-', linewidth=1, alpha=0.5)
    
    # Add value labels on bars
    def autolabel(bars):
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.1f}x',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
    
    autolabel(bars2)
    autolabel(bars3)
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/flash_attention_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

create_summary_comparison()

## Summary

**Key Takeaways**:

1. **Problem**: Standard attention has O(N²) memory complexity, limiting sequence length

2. **Solution**: Flash Attention uses tiling and recomputation to achieve:
   - O(N) memory complexity
   - 2-8x speedup
   - Exact same results as standard attention

3. **Algorithm**: Blocks Q, K, V into tiles, processes in SRAM, recomputes in backward pass

4. **Versions**:
   - v1 (2022): Original, 2-4x speedup
   - v2 (2023): Optimized, 4-8x speedup

5. **Integration**: Easy to use with:
   - Custom PyTorch models
   - Hugging Face Transformers (built-in support)
   - Most modern LLMs

6. **Best For**:
   - Long sequences (> 512 tokens)
   - Memory-constrained training
   - Large-scale transformer models

**Papers**:
- Flash Attention v1: [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135) (Dao et al., 2022)
- Flash Attention v2: [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691) (Dao, 2023)